# Portfolio Analysis - 1.0x Long

This notebook runs a 1.0x leverage long portfolio backtest using the modularized analysis engine.

The analysis logic is now centralized in `portfolio_engine.py` and signal configurations are in `portfolio_config.py`, eliminating code duplication with the 1.5x long portfolio.

## Setup & Configuration

In [ ]:
# Import modular components
from portfolio_config import get_signal_config
from portfolio_engine import PortfolioAnalysisEngine
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# Get configuration for this portfolio variant
config = get_signal_config('1.0x_long')

print(f"Portfolio: {config['name']}")
print(f"Leverage: {config['leverage']}x")
print(f"Number of ETFs: {len(config['etf_universe'])}")
print(f"Sample ETFs: {config['etf_universe'][:5]}")

## Run Portfolio Analysis

In [ ]:
# Initialize the analysis engine with this portfolio's configuration
engine = PortfolioAnalysisEngine(config)

# Run the complete analysis pipeline
# This loads data, generates signals, runs backtest, and calculates metrics
results = engine.run_full_analysis()

print("Analysis complete!")

## Performance Metrics

In [ ]:
# Extract metrics from results
metrics = results['metrics']
backtest = results['backtest']

# Display key metrics
print("\n=== Performance Metrics ===")
print(f"Sharpe Ratio:        {metrics.get('sharpe_ratio', 'N/A'):.4f}")
print(f"Sortino Ratio:       {metrics.get('sortino_ratio', 'N/A'):.4f}")
print(f"Max Drawdown:        {metrics.get('max_drawdown', 'N/A'):.4f}")
print(f"Total Return:        {metrics.get('total_return', 'N/A'):.4f}")
print(f"Win Rate:            {metrics.get('win_rate', 'N/A'):.4f}")
print(f"Profit Factor:       {metrics.get('profit_factor', 'N/A'):.4f}")

# Create metrics DataFrame for easier viewing
metrics_df = pd.DataFrame(metrics, index=[config['name']])
metrics_df

## Equity Curve

In [ ]:
# Extract equity curve data
equity_curve = backtest.get('equity_curve', None)

if equity_curve is not None:
    plt.figure(figsize=(14, 6))
    plt.plot(equity_curve.index, equity_curve.values, linewidth=2, label=config['name'])
    plt.title(f"Portfolio Equity Curve - {config['name']}", fontsize=14, fontweight='bold')
    plt.xlabel("Date", fontsize=12)
    plt.ylabel("Portfolio Value ($)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Equity curve data not available")

## Drawdown Analysis

In [ ]:
# Calculate drawdown from equity curve
if equity_curve is not None:
    running_max = equity_curve.expanding().max()
    drawdown = (equity_curve - running_max) / running_max * 100
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))
    
    # Equity curve
    ax1.plot(equity_curve.index, equity_curve.values, color='blue', linewidth=2)
    ax1.fill_between(equity_curve.index, equity_curve.values, alpha=0.3, color='blue')
    ax1.set_title(f"Equity Curve - {config['name']}", fontsize=12, fontweight='bold')
    ax1.set_ylabel("Portfolio Value ($)", fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Drawdown
    ax2.fill_between(drawdown.index, drawdown.values, color='red', alpha=0.5)
    ax2.plot(drawdown.index, drawdown.values, color='darkred', linewidth=1)
    ax2.set_title("Drawdown (%)", fontsize=12, fontweight='bold')
    ax2.set_xlabel("Date", fontsize=10)
    ax2.set_ylabel("Drawdown (%)", fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"Maximum Drawdown: {drawdown.min():.2f}%")
else:
    print("Equity curve data not available for drawdown analysis")

## Monthly Returns

In [ ]:
# Calculate daily returns
if equity_curve is not None:
    daily_returns = equity_curve.pct_change().dropna()
    
    # Aggregate to monthly returns
    monthly_returns = equity_curve.resample('M').last().pct_change().dropna()
    
    # Create monthly returns table
    monthly_df = pd.DataFrame({
        'Month': monthly_returns.index.strftime('%Y-%m'),
        'Return (%)': (monthly_returns.values * 100).round(2)
    })
    
    print("\nMonthly Returns:")
    print(monthly_df.to_string(index=False))
    
    # Visualize monthly returns
    plt.figure(figsize=(14, 6))
    colors = ['green' if x > 0 else 'red' for x in monthly_returns.values]
    plt.bar(range(len(monthly_returns)), monthly_returns.values * 100, color=colors, alpha=0.7)
    plt.title(f"Monthly Returns - {config['name']}", fontsize=14, fontweight='bold')
    plt.xlabel("Month", fontsize=12)
    plt.ylabel("Return (%)", fontsize=12)
    plt.xticks(range(0, len(monthly_returns), 12), 
              [monthly_returns.index[i].strftime('%Y') if i % 12 == 0 else '' 
               for i in range(len(monthly_returns))])
    plt.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
else:
    print("Equity curve data not available for returns analysis")

## Returns Distribution

In [ ]:
# Analyze distribution of returns
if equity_curve is not None:
    returns = equity_curve.pct_change().dropna()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    axes[0].hist(returns.values * 100, bins=50, alpha=0.7, color='blue', edgecolor='black')
    axes[0].axvline(returns.mean() * 100, color='red', linestyle='--', linewidth=2, label=f'Mean: {returns.mean()*100:.3f}%')
    axes[0].set_title("Daily Returns Distribution", fontsize=12, fontweight='bold')
    axes[0].set_xlabel("Daily Return (%)", fontsize=10)
    axes[0].set_ylabel("Frequency", fontsize=10)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Q-Q plot (to check normality)
    from scipy import stats
    stats.probplot(returns.values, dist="norm", plot=axes[1])
    axes[1].set_title("Q-Q Plot (Normality Check)", fontsize=12, fontweight='bold')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nDaily Returns Statistics:")
    print(f"Mean:       {returns.mean()*100:.4f}%")
    print(f"Std Dev:    {returns.std()*100:.4f}%")
    print(f"Skewness:   {returns.skew():.4f}")
    print(f"Kurtosis:   {returns.kurtosis():.4f}")
    print(f"Min:        {returns.min()*100:.4f}%")
    print(f"Max:        {returns.max()*100:.4f}%")
else:
    print("Equity curve data not available for distribution analysis")

## Signal Positions Over Time

In [ ]:
# Plot positions for a sample of ETFs
positions = backtest.get('positions', None)

if positions is not None:
    # Display positions for top 5 ETFs
    sample_etfs = config['etf_universe'][:5]
    
    fig, axes = plt.subplots(len(sample_etfs), 1, figsize=(14, 3*len(sample_etfs)), sharex=True)
    if len(sample_etfs) == 1:
        axes = [axes]
    
    for ax, etf in zip(axes, sample_etfs):
        if etf in positions.columns:
            ax.bar(positions.index, positions[etf], alpha=0.7, color='steelblue')
            ax.set_title(f"Position Size - {etf}", fontsize=11, fontweight='bold')
            ax.set_ylabel("Shares", fontsize=10)
            ax.grid(True, alpha=0.3, axis='y')
            ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    axes[-1].set_xlabel("Date", fontsize=10)
    plt.tight_layout()
    plt.show()
else:
    print("Position data not available")

## Summary Statistics

In [ ]:
# Create comprehensive summary
print("\n" + "="*60)
print(f"PORTFOLIO ANALYSIS SUMMARY - {config['name']}")
print("="*60)

print(f"\nConfiguration:")
print(f"  Leverage:              {config['leverage']}x")
print(f"  Number of ETFs:        {len(config['etf_universe'])}")
print(f"  Bond ETFs included:    {len(config['bond_etfs'])}")

print(f"\nPerformance:")
for key, value in metrics.items():
    if isinstance(value, float):
        print(f"  {key.replace('_', ' ').title():<25} {value:>10.6f}")
    else:
        print(f"  {key.replace('_', ' ').title():<25} {str(value):>10}")

print(f"\n{'='*60}")
print("Analysis completed successfully!")
print("\nNote: This notebook now uses the modularized portfolio_engine.py")
print("To compare with other variants, simply change the config:")
print("  config = get_signal_config('1.5x_long')  # For 1.5x variant")
print("="*60)